# Landscript: Glyph Extraction Pipeline

Discovers letter-like shapes in satellite imagery of Bangalore.

1. Mounts Google Drive for persistent storage
2. Installs the landscript package from GitHub
3. Authenticates Earth Engine
4. Searches for Sentinel-2 scenes
5. Downloads and tiles imagery
6. Runs CV pipeline to find glyph candidates
7. Saves glyphs + metadata

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/landscript'
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
%cd /content

In [ ]:
# @title 2. Install landscript package
# Install from GitHub (main branch)
!pip install git+https://github.com/pkddata/landscript.git

# Force reinstall for development (uncomment if needed)
# !pip install --upgrade --force-reinstall git+https://github.com/pkddata/landscript.git

# Additional Colab-specific deps
!pip install requests pillow tqdm

In [ ]:
# @title 3. Authenticate Earth Engine
import ee
from google.colab import auth
auth.authenticate_user()
ee.Authenticate()
ee.Initialize()
print('Earth Engine ready!')

In [ ]:
# @title 4. Configure pipeline for Bangalore
from pathlib import Path
from landscript.config import REGIONS, PipelineConfig

# data_dir stays in the repo (glyphs + metadata get committed)
# drive_dir stores source imagery and tiles (too large for git)
cfg = PipelineConfig(
    bbox=REGIONS["bangalore"],
    region_name="bangalore",
    state="Karnataka",
    satellite="sentinel-2",
    data_dir=Path('/content/landscript/data'),
    drive_dir=Path(f'{DRIVE_ROOT}/data'),
    cloud_cover_max=20,
    similarity_threshold=0.15,
)
print(f'Region: {cfg.region_name}')
print(f'Satellite: {cfg.satellite}')
print(f'Font: {cfg.font.family}')
print(f'Bounds: {cfg.bbox}')
print(f'Repo data dir: {cfg.data_dir}')
print(f'Drive bulk dir: {cfg.drive_dir}')

In [ ]:
# @title 5. List available Sentinel-2 scenes
from landscript.gee import list_scenes
scenes = list_scenes(cfg)
print(f'Found {len(scenes)} scenes')
for s in scenes[:10]:
    print(f"  {s['date'][:10]} | cloud: {s['cloud']:.0f}% | satellite: {s['satellite']}")

In [ ]:
# @title 6. Download a scene as tiles
from landscript.gee import get_sentinel_collection, export_rgb_tile
import ee

collection = get_sentinel_collection(cfg)
scene_list = collection.limit(5).toList(5)

region = ee.Geometry.Rectangle([
    cfg.bbox.min_lon, cfg.bbox.min_lat,
    cfg.bbox.max_lon, cfg.bbox.max_lat
])

for i in range(scene_list.size().getInfo()):
    img = ee.Image(scene_list.get(i))
    date = img.date().format('YYYY-MM-dd').getInfo()
    fname = f"{cfg.region_name}_{date}.tif"
    out = cfg.source_dir / fname
    print(f'Downloading {fname}...')
    result = export_rgb_tile(img, region, out, cfg)
    if result:
        print(f'  Saved: {result}')
    else:
        print(f'  Failed!')

In [ ]:
# @title 7. Split source imagery into tiles
import rasterio
from rasterio.windows import Window
from pathlib import Path

source_files = sorted(cfg.source_dir.glob('*.tif'))
print(f'Tiling {len(source_files)} source image(s)...')

for src_path in source_files:
    with rasterio.open(src_path) as src:
        width, height = src.width, src.height
        print(f'  {src_path.name}: {width}x{height}')
        tid = 0
        for y in range(0, height, cfg.tile_size):
            for x in range(0, width, cfg.tile_size):
                w = min(cfg.tile_size, width - x)
                h = min(cfg.tile_size, height - y)
                window = Window(x, y, w, h)
                tile = src.read(window=window)
                if tile.shape[1] < cfg.tile_size or tile.shape[2] < cfg.tile_size:
                    continue
                tile_path = cfg.tiles_dir / f"{src_path.stem}_tile{tid:04d}.png"
                # Save as PNG (RGB, 8-bit)
                import numpy as np
                img = np.moveaxis(tile[:3], 0, -1)
                img = (img / img.max() * 255).astype(np.uint8) if img.max() > 0 else img.astype(np.uint8)
                import cv2
                cv2.imwrite(str(tile_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
                tid += 1
        print(f'    Created {tid} tiles')

In [ ]:
# @title 8. Run CV pipeline to find glyphs
from landscript.cv_pipeline import (
    load_letter_templates, to_grayscale, apply_threshold,
    find_contours, filter_contours, match_shape,
    contour_to_polygon, extract_glyph_crop,
)
from landscript.metadata import GlyphStore
import cv2
import numpy as np

print('Loading letter templates...')
templates = load_letter_templates(cfg)
print(f'Loaded {len(templates)} letter templates')

store = GlyphStore(cfg.metadata_dir / 'glyphs.json')
print(f'Existing glyphs in store: {store.count()}')

tile_files = sorted(cfg.tiles_dir.glob('*.png'))
print(f'Processing {len(tile_files)} tiles...')

candidates_found = 0
for tile_path in tile_files:
    img = cv2.imread(str(tile_path))
    if img is None:
        continue

    gray = to_grayscale(img)
    binary = apply_threshold(gray, cfg.threshold_method)
    contours = find_contours(binary)
    contours = filter_contours(contours, cfg)

    for ci, contour in enumerate(contours):
        poly = contour_to_polygon(contour, cfg.epsilon)
        for tmpl in templates:
            score = match_shape(poly, tmpl.contour)
            if score >= cfg.similarity_threshold:
                continue

            x, y, w, h = cv2.boundingRect(contour)
            glyph_id = store.add({
                "letter": tmpl.letter,
                "score": float(score),
                "bbox": {"x": x, "y": y, "w": w, "h": h},
                "source_tile": tile_path.name,
                "region": cfg.region_name,
                "state": cfg.state,
                "country": cfg.country,
            })
            glyph_path = cfg.glyphs_dir / tmpl.letter / f"{glyph_id}.png"
            extract_glyph_crop(img, contour, glyph_path)
            candidates_found += 1

print(f'Done! {candidates_found} glyph candidates saved.')

In [ ]:
# @title 9. Browse results by letter
letter = 'A'
results = store.search(letter=letter, limit=20)
print(f"Found {len(results)} candidates for '{letter}'")
for r in results[:10]:
    print(f"  {r['id']}: score={r['score']:.4f} | accepted={r['accepted']}")
    glyph_path = cfg.glyphs_dir / r['letter'] / f"{r['id']}.png"
    if glyph_path.exists():
        from IPython.display import display, Image as IPImage
        display(IPImage(filename=str(glyph_path)))

In [ ]:
# @title 10. Accept/curate a glyph
glyph_id = "glyph_20260101_120000_123456"  # Replace with actual ID
store.accept(glyph_id)
print(f'Accepted: {glyph_id}')

In [ ]:
# @title 11. Commit new glyphs to the repo
!git -C /content/landscript add data/glyphs/ data/metadata/ fonts/
!git -C /content/landscript commit -m "Add glyphs and metadata" || echo 'Nothing new to commit'
!git -C /content/landscript push
print('Pushed to GitHub!')

In [ ]:
# @title Summary stats
all_glyphs = store.all()
accepted = [g for g in all_glyphs if g.get('accepted')]
letters_found = set(g.get('letter') for g in all_glyphs if g.get('letter'))
print(f'Total glyphs: {len(all_glyphs)}')
print(f'Accepted: {len(accepted)}')
print(f'Unique letters: {sorted(letters_found)}')